# Explore
This notebook shows how the code works

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from pcr import storm, helper, shoreline, slr
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

In [ ]:
import importlib
importlib.reload(storm)
importlib.reload(helper)
importlib.reload(slr)
importlib.reload(shoreline)

In [ ]:
# set up a seed 
np.random.seed(42)

## Storm Detect
detecting storm from historical wave condition

In [ ]:
# test out the function using data from data/wave_srilanka.csv
wave_data = pd.read_csv('../data/wave_srilanka.csv')

# get hs, dir, tp, time from dataframe
time = wave_data.iloc[:, 0].values
hs = wave_data.iloc[:, 1].values
dir = wave_data.iloc[:, 2].values
tp = wave_data.iloc[:, 3].values
ts_hs = 95
ts_dur = 12.0

# detect storms 
storms, storms_ts = storm.detect(hs, dir, tp, time, ts_hs, ts_dur)

In [ ]:
# convert to datetime
time_datetime = helper.datenum_to_datetime(time)
wave_df = pd.DataFrame({
    'time': time_datetime, 
    'hs': hs, 
    'dir': dir, 
    'tp': tp
}) 

storms_ts['time_datetime'] = storms_ts['time'].apply(helper.datenum_to_datetime)
storms['end_datetime'] = storms['end'].apply(helper.datenum_to_datetime)

fig = plt.figure() 
plt.plot(time_datetime, hs, label='Significant wave height (m)') 

for storm_id in storms_ts['storm_id'].unique(): 
    storm_single = storms_ts[storms_ts['storm_id'] == storm_id]
    plt.plot(storm_single['time_datetime'], storm_single['hs'], color='red')

plt.scatter(storms_ts.groupby('storm_id').last()['time_datetime'], storms_ts.groupby('storm_id').last()['hs'], color='green', marker='*') 

plt.xlabel('time') 
plt.ylabel('Hs (m)') 
plt.ylim((0, np.max(hs)*1.1)) 
plt.grid() 
plt.show()

Make a one-to-one comparison with Matlab output

In [ ]:
# load mat file into pandas DataFrame
storms_mat = helper.load_mat_to_df('../data/storms_95_12.mat', 'Storm')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Duration Comparison', 'Hs Max Comparison', 'Tp Mean Comparison', 'Gap Comparison'),
    x_title='Python Data (storms)',
    y_title='Matlab Data (storms_mat)',
    vertical_spacing=0.1, horizontal_spacing=0.1
)

# 1st subplot 
fig.add_trace(
    go.Scatter(x=[storms['duration'].min(), storms['duration'].max()], y=[storms['duration'].min(), storms['duration'].max()], mode='lines', showlegend=False, line=dict(color='grey', dash='dash')), 
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=storms['duration'], y=storms_mat['Duration'], mode='markers', name='Duration'), 
    row=1, col=1
)

# 2nd subplot 
fig.add_trace(
    go.Scatter(x=[storms['hs_max'].min(), storms['hs_max'].max()], y=[storms['hs_max'].min(), storms['hs_max'].max()], mode='lines', showlegend=False, line=dict(color='grey', dash='dash')), 
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=storms['hs_max'], y=storms_mat['HsMax'], mode='markers', name='HS Max'),
    row=1, col=2
)

# 3rd subplot 
fig.add_trace(
    go.Scatter(x=[storms['tp_mean'].min(), storms['tp_mean'].max()], y=[storms['tp_mean'].min(), storms['tp_mean'].max()], mode='lines', showlegend=False, line=dict(color='grey', dash='dash')), 
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=storms['tp_mean'], y=storms_mat['TpMean'], mode='markers', name='Tp Mean'),
    row=2, col=1
)

# 4th subplot
fig.add_trace(
    go.Scatter(x=[storms['gap'].min(), storms['gap'].max()], y=[storms['gap'].min(), storms['gap'].max()], mode='lines', showlegend=False, line=dict(color='grey', dash='dash')), 
    row=2, col=2
)

fig.add_trace(
    go.Scatter(x=storms['gap'], y=storms_mat['Gap'], mode='markers', name='Gap'),
    row=2, col=2
)

# Update layout
fig.update_layout(
    title_text='Comparison of Strom Detection Variables Between Python and Matlab',
    height=800, width=800,
    showlegend=True
)

# Display the figure
fig.show()

In [ ]:
# Check the conditions
condition_1 = np.sum(storms['duration'].values - storms_mat['Duration'].values) < 1e-6
condition_2 = np.sum(storms['hs_max'].values - storms_mat['HsMax'].values) < 1e-6
condition_3 = np.sum(storms['tp_mean'].values - storms_mat['TpMean'].values) < 1e-6
condition_4 = np.sum(storms['gap'].values - storms_mat['Gap'].values) < 1e-6

# Check if all conditions are met
if condition_1 and condition_2 and condition_3 and condition_4:
    print('All conditions are met! The detected storm in python is consistent with matlab.')
else:
    print('Conditions are not fully met.')


## fit and generate storms
Fit the detecting storms to distributions and do a random sampling from the function


In [ ]:
np.random.seed(42)

nr_sample = 1000 # 60,000,000

fitted_storms = storm.fit_storm(storms)
fitted_gap = storm.fit_gap_monsoon(storms)
max_dur = np.max(storms.duration)
storms_sample = storm.generate(fitted_storms, nr_sample, oversample=0.1, max_dur=max_dur)
storms_sample = storm.sampling_gap_ecdf(fitted_gap, storms_sample)

Check fitting of the hs and duration if it is the same with matlab output

In [ ]:
pgev_hs_matlab = (-0.3087, 1.4850, 0.0988)
pgev_dur_matlab = (-0.6505, 31.3017, 16.8024)

In [ ]:
import scipy.stats as stat

# 1. fit to GEV distribution 
pgev_hs = fitted_storms['gev_hs']
pgev_dur = fitted_storms['gev_dur']
hs = storms['hs_max']
durations = storms['duration']

# 2. Generate the fitted CDF from the GEV distribution
x_dur = np.linspace(min(durations), max(durations), 1000)
fitted_cdf_dur = stat.genextreme.cdf(x_dur, pgev_dur[0], loc=pgev_dur[1], scale=pgev_dur[2])
fitted_cdf_dur_m = stat.genextreme.cdf(x_dur, pgev_dur_matlab[0], loc=pgev_dur_matlab[1], scale=pgev_dur_matlab[2])

x_hs = np.linspace(min(hs), max(hs), 1000)
fitted_cdf_hs = stat.genextreme.cdf(x_hs, pgev_hs[0], loc=pgev_hs[1], scale=pgev_hs[2])
fitted_cdf_hs_m = stat.genextreme.cdf(x_hs, pgev_hs_matlab[0], loc=pgev_hs_matlab[1], scale=pgev_hs_matlab[2])

# 3. Compute the empirical CDF
sorted_data_dur, empirical_cdf_dur = storm.empirical_cdf(durations)
sorted_data_hs, empirical_cdf_hs = storm.empirical_cdf(hs)

# plot the figure
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[1].plot(x_hs, fitted_cdf_hs, label='Fitted python hs', color='red')
axs[1].plot(x_hs, fitted_cdf_hs_m, label='Fitted python hs', color='green', linestyle='--')
axs[1].plot(sorted_data_hs, empirical_cdf_hs, label='Data hs', color='blue')
axs

axs[0].plot(x_dur, fitted_cdf_dur, label='Fitted python duration', color='red')
axs[0].plot(x_dur, fitted_cdf_dur_m, label='Fitted matlab hs', color='green', linestyle='--')
axs[0].plot(sorted_data_dur, empirical_cdf_dur, label='Data duration', color='blue')

axs[0].legend()
axs[1].legend()

axs[0].grid()
axs[1].grid()

## Sampling 
compare the distribution of sampled Hs, Dur, Tp and Gap(?)

In [ ]:
import mat73

# n_matlab = 68,049,460
data_dict = mat73.loadmat('../data/test/diagnostic_storm.mat')
storm_track = data_dict['storm_track'] # day, hs, duration, tp

temp = storm_track #[0:nr_sample]
# storms_sample = storms_sample.iloc[:nr_sample]

storms_sample_mat = pd.DataFrame({
    'hs': temp[:, 1],
    'duration': temp[:, 2],
    'tp': temp[:, 3],
    'day': temp[:, 0]
})

In [ ]:
# fig, axs = plt.subplots(ncols=3, nrows=1, figsize=(12,5))#, layout='constrained')

# fig.suptitle('Distribution of Sampled Storms', fontsize=14)

# axs[0].hist(storms_sample['hs'], bins=100, label='python')
# axs[0].hist(storms_sample_mat['hs'], bins=100, label='matlab')
# axs[0].set_title('Significant Wave Height')
# axs[0].set_xlim(0,5)

# axs[1].hist(storms_sample['duration'], bins=100, label='python')
# axs[1].hist(storms_sample_mat['duration'], bins=100, label='matlab')
# axs[1].set_title('Duration')

# axs[2].hist(storms_sample['tp'], bins=100, label='python')
# axs[2].hist(storms_sample_mat['tp'], bins=100, label='matlab')
# axs[2].set_title('Peak Period')

# axs[1,1].hist(storms_sample['gap'], bins=100)
# axs[1,1].set_title('Gap')

# fig.legend(['Python', 'Matlab'])

print('Comparison of sampled storms in mean, median, 95th percentile, 99th percentile (n=60,000,000)')

print('hs python: {:.3f}, {:.3f}, {:.3f}, {:.3f}'.format(
    np.mean(storms_sample['hs']), 
    np.median(storms_sample['hs']), 
    np.percentile(storms_sample['hs'], 95), 
    np.percentile(storms_sample['hs'], 99.99))
)
print('hs matlab: {:.3f}, {:.3f}, {:.3f}, {:.3f}'.format(
    np.mean(storms_sample_mat['hs']), 
    np.median(storms_sample_mat['hs']), 
    np.percentile(storms_sample_mat['hs'], 95), 
    np.percentile(storms_sample['hs'], 99.99))
)

print('duration python: {:.3f}, {:.3f}, {:.3f}, {:.3f}'.format(
    np.mean(storms_sample['duration']), 
    np.median(storms_sample['duration']), 
    np.percentile(storms_sample['duration'], 95), 
    np.percentile(storms_sample['duration'], 99.99))
)
print('duration matlab: {:.3f}, {:.3f}, {:.3f}, {:.3f}'.format(
    np.mean(storms_sample_mat['duration']), 
    np.median(storms_sample_mat['duration']), 
    np.percentile(storms_sample_mat['duration'], 95),
    np.percentile(storms_sample_mat['duration'], 99.99),)
)

print('tp python: {:.3f}, {:.3f}, {:.3f}, {:.3f}'.format(
    np.mean(storms_sample['tp']), 
    np.median(storms_sample['tp']), 
    np.percentile(storms_sample['tp'], 95),
    np.percentile(storms_sample['tp'], 99.99))
)
print('tp matlab: {:.3f}, {:.3f}, {:.3f}, {:.3f}'.format(
    np.mean(storms_sample_mat['tp']), 
    np.median(storms_sample_mat['tp']), 
    np.percentile(storms_sample_mat['tp'], 95), 
    np.percentile(storms_sample_mat['tp'], 99.99))
)



## Simulation
Storm simulation: simulating a storm from start to end

In [ ]:
# start date and end date 
date_start = datetime(
    year=2000, 
    month=1, 
    day=1
)

date_end = datetime(
    year=2100,
    month=12, 
    day=31
)

In [ ]:
# get the index of the start
start_idx = np.where(storms_sample_mat['day'] == 0)

# 1000 storms 
storms_sample_matinput = storms_sample_mat.iloc[:start_idx[0][10500]].copy()

In [ ]:
# calculate end days and gap 
day_start = storms_sample_matinput['day']
end_days = day_start + (storms_sample_matinput['duration'] / 24)

prev_day_end = np.concatenate([[day_start[0]], end_days[:-1]]) # leaving the first storm has 0 gap

# calculate gap and remove gap with lower than 0
gap_days =  day_start - prev_day_end 
gap_days[gap_days < 0] = 0

storms_sample_matinput['gap_data'] = gap_days

# sample gap
# storms_sample_matinput = storm.sampling_gap_ecdf(fitted_gap, storms_sample_matinput)

# populate the direction first 
storms_sample_matinput['direction'] = 0 


In [ ]:
storm_count = 0
day_starts = []

for i in range(10000):
    
    ts_temp = storm.generate_monsoon_ts(
        date_start=date_start, 
        date_end=date_end, 
        storms_sample=storms_sample_matinput, 
        fitted_gap=fitted_gap, 
        start_storm=storm_count
    )

    storm_count += len(ts_temp)

    day_starts.append(ts_temp['day_start'].values)



In [ ]:
python_start = np.concatenate(day_starts)

storms_sample_matinput['day_start_python'] = np.nan
storms_sample_matinput.loc[:len(python_start)-1, 'day_start_python'] = python_start

storms_sample_matinput['day_start_mat'] = storms_sample_matinput['day']

In [ ]:
print('number of storm for 10,000 simulation in matlab:', start_idx[0][10000])
print('number of storm for 10,000 simulation in python:', python_start.shape[0])

In [ ]:
(python_start.shape[0] - start_idx[0][10000])/10000

1000 simulation in python require 683,223 storms while in matlab 669,721 storms 

10,000 simulation in python require 6,830,766 storms while in matlab 6,695,320 storms

In [ ]:
print('lambda year in py: {:.4f}'.format(fitted_gap['lambda_year']))
print('lambda year in mat: 363.4397')
print('lambda season in py: {:.4f}'.format(fitted_gap['lambda_season']))
print('lambda season in mat: 71.1293')


In [ ]:
gap_mat = np.loadtxt('../data/test/gap.csv')
p_gap_mat = np.loadtxt('../data/test/p_gap.csv')
gap_py = fitted_gap['ecdf_gap'][0]
p_gap_py = fitted_gap['ecdf_gap'][1]

plt.plot(gap_mat, p_gap_mat, label='mat')
plt.plot(gap_py, p_gap_py, label='py', linestyle=':', color='red')
plt.xlabel('gap (days)')


In [ ]:
gaps = fitted_gap['ecdf_gap'][0]
p_gap = fitted_gap['ecdf_gap'][1]

p_below = p_gap[np.where(gaps >= 1)[0][0]]
r = np.random.uniform(size=10000000) * (1-p_below) + p_below

gap_test = storm.sample_ecdf(fitted_gap['ecdf_gap'], r=r)

np.mean(gap_test)

In [ ]:
# manipulate gap so that it is the same with matlab

storms_sample_matinput['gap_sample_py'] = storms_sample_matinput['gap']
storms_sample_matinput['gap'] = storms_sample_matinput['gap_data']

# shift the gap, then fill the gap between seaon (> 100) to the average of gap 
print('average gap: {:.3f}'.format(np.interp(0.5, fitted_gap['ecdf_gap'][1], fitted_gap['ecdf_gap'][0])))
a = np.concat([storms_sample_matinput['gap'].loc[1:].values, [5.75]])

storms_sample_matinput['gap'] = a
storms_sample_matinput.loc[storms_sample_matinput['gap'] > 100, 'gap'] = 5.75

# check df
storms_sample_matinput.head()

In [ ]:
storm_count = 0
day_starts_mat = []

for i in range(10000):
    
    ts_temp = storm.generate_monsoon_ts(
        date_start=date_start, 
        date_end=date_end, 
        storms_sample=storms_sample_matinput, 
        fitted_gap=fitted_gap, 
        start_storm=storm_count
    )

    storm_count += len(ts_temp)

    day_starts_mat.append(ts_temp['day_start'].values)


In [ ]:
storms_sample_matinput[['day_start_mat', 'day_start_python_matinput']].head(10)

In [ ]:
python_start_matinput = np.concatenate(day_starts_mat)

storms_sample_matinput['day_start_python_matinput'] = np.nan
storms_sample_matinput.loc[:len(python_start_matinput)-1, 'day_start_python_matinput'] = python_start_matinput


In [ ]:
day_start_py = storms_sample_matinput['day_start_python'][~np.isnan(storms_sample_matinput['day_start_python'])]
year_py = [(date_start + timedelta(days=day)).year for day in day_start_py]

storms_sample_matinput['year_py'] = np.nan
storms_sample_matinput.loc[:len(year_py)-1, 'year_py'] = year_py

In [ ]:
year_mat = [(date_start + timedelta(days=day)).year for day in storms_sample_matinput['day_start_mat']]
storms_sample_matinput['year_mat'] = year_mat

In [ ]:
yearly_py = storms_sample_matinput.groupby(by='year_py').hs.count().mean()/10000
yearly_mat = storms_sample_matinput.groupby(by='year_mat').hs.count().mean()/10500

In [ ]:
print('mean of yearly storm in py (10,000 simulation): {:.3f}'.format(yearly_py))
print('mean of yearly storm in mat (10,500 simulation): {:.3f}'.format(yearly_mat))

In [ ]:
storms_sample_matinput.head()

In [ ]:
# try bin the month 
day_start_py = storms_sample_matinput['day_start_python']
end_days_py = day_start_py + (storms_sample_matinput['duration'] / 24)

prev_day_end_py = np.concatenate([[day_start_py[0]], end_days_py[:-1]]) # leaving the first storm has 0 gap

# calculate gap and remove gap with lower than 0
gap_days_py =  day_start_py - prev_day_end_py
gap_days_py[gap_days_py < 0] = 0

storms_sample_matinput['gap_py'] = gap_days_py

In [ ]:
# try bin the month 
storms_sample_matinput.loc[20:40]

In [ ]:
# simulate python
synthetic_storm = storm.generate_monsoon_ts(date_start=date_start, date_end=date_end, storms_sample=storms_sample, fitted_gap=fitted_gap)

# load generated storm from matlab
synthetic_storm_m = helper.load_mat_to_df('../data/simulation_storm_m.mat', 'synthetic_matlab')

synthetic_storm_m = synthetic_storm_m.dropna()
synthetic_storm_m = synthetic_storm_m.rename(columns={
    'dur': 'duration',
    'start': 'day_start',
    'end': 'day_end',
    'del_x': 'del_x_mat'
    })

synthetic_storm_m['day_end'] = synthetic_storm_m['day_start'] + synthetic_storm_m['duration']/24

ends = np.concatenate([[0], synthetic_storm_m['day_end'][:-1]])
synthetic_storm_m['gap'] = synthetic_storm_m['day_start'].values - ends

In [ ]:
# sort data for peak period
synth_hs, synth_p_ecdf = storm.empirical_cdf(synthetic_storm['hs'])
synth_hs_m, synth_p_ecdf_m = storm.empirical_cdf(synthetic_storm_m['hs'])

synth_dur, _ = storm.empirical_cdf(synthetic_storm['duration'])
synth_dur_m, _ = storm.empirical_cdf(synthetic_storm_m['duration'])

synth_tp, _ = storm.empirical_cdf(synthetic_storm['tp'])
synth_tp_m, _ = storm.empirical_cdf(synthetic_storm_m['tp'])

synth_gap, _ = storm.empirical_cdf(synthetic_storm['gap'])
synth_gap_m, _ = storm.empirical_cdf(synthetic_storm_m['gap'])

fig, axs = plt.subplots(ncols=2, nrows=2, figsize=(9,6), layout='constrained')

axs[0,0].scatter(synth_hs, synth_p_ecdf, marker='x', alpha=0.8)
axs[0,0].scatter(synth_hs_m, synth_p_ecdf_m, marker='x', alpha=0.8)
axs[0,0].set_title('Hs (m)')
# axs[0,0].set_xlim(1,4.5)

axs[0,1].scatter(synth_dur, synth_p_ecdf, marker='x')
axs[0,1].scatter(synth_dur_m, synth_p_ecdf_m, marker='x')
axs[0,1].set_title('Duration (h)')
# axs[1].set_ylabel('cdf')
# axs[1].set_xlim(sorted_data_dur[0], sorted_data_dur[-1])

axs[1,0].scatter(synth_tp, synth_p_ecdf, marker='x')
axs[1,0].scatter(synth_tp_m, synth_p_ecdf_m, marker='x')
axs[1,0].set_title('Tp (s)')
# axs[2].plot(sorted_data_tp, empirical_cdf_tp, label='Data hs', color='blue')
# axs[2].set_xlim(sorted_data_tp[0], sorted_data_tp[-1])

axs[1,1].scatter(synth_gap, synth_p_ecdf, marker='x')
axs[1,1].scatter(synth_gap_m, synth_p_ecdf_m, marker='x')
axs[1,1].set_title('Gap (days)')

fig.suptitle('Empirical CDF of Sampled Storms (1 simulation)')
axs[1,1].legend(['Python', 'Matlab'])

In [ ]:
import plotly.graph_objects as go 
from plotly.subplots import make_subplots

# convert days to datetime
synthetic_storm['start'] = [date_start + pd.Timedelta(something, 'D') for something in synthetic_storm['day_start']]
synthetic_storm_m['date_start'] = [date_start + pd.Timedelta(something, 'D') for something in synthetic_storm_m['day_start']]

# fig = px.scatter(
#     synthetic_storm, 
#     x='start', 
#     y='hs',
# )

# fig.show()

# Create subplots
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    vertical_spacing=0.05)

# Plot 1: Hs Plot (scatter plot)
fig.add_trace(go.Scatter(x=synthetic_storm['start'], 
                         y=synthetic_storm['hs'], 
                         mode='markers', 
                         name='Python', 
                         marker=dict(symbol='x', color='red', size=8), 
                         legendgroup='Python'),
              row=1, col=1)

fig.add_trace(go.Scatter(x=synthetic_storm_m['date_start'], 
                         y=synthetic_storm_m['hs'],
                         mode='markers', 
                         name='Matlab', 
                         marker=dict(symbol='x', color='blue', size=8), 
                         legendgroup='Matlab'), 
                row=1, col=1)

fig.update_yaxes(title_text='Hs (m)', row=1, col=1)

# Plot 2: Tp Plot (scatter plot)
fig.add_trace(go.Scatter(x=synthetic_storm['start'], 
                         y=synthetic_storm['tp'], 
                         mode='markers',
                         name='Python',
                         marker=dict(symbol='x', color='red', size=8), 
                         legendgroup='Python', 
                         showlegend=False),
              row=2, col=1)

fig.add_trace(go.Scatter(x=synthetic_storm_m['date_start'], 
                         y=synthetic_storm_m['tp'], 
                         mode='markers',
                         name='Matlab',
                         marker=dict(symbol='x', color='blue', size=8),
                         legendgroup='Matlab',
                         showlegend=False), 
              row=2, col=1)

fig.update_yaxes(title_text='Tp (s)', row=2, col=1)

# Plot 3: Dur Plot (scatter plot with x and y axis labels)
fig.add_trace(go.Scatter(x=synthetic_storm['start'], 
                         y=synthetic_storm['duration'], 
                         mode='markers',
                         name='Python',
                         marker=dict(symbol='x', color='red', size=8),
                         legendgroup='Python',
                         showlegend=False),
              row=3, col=1)

fig.add_trace(go.Scatter(x=synthetic_storm_m['date_start'], 
                         y=synthetic_storm_m['duration'], 
                         mode='markers',
                         name='Matlab',
                         marker=dict(symbol='x', color='blue', size=8),
                         legendgroup='Matlab',
                         showlegend=False), 
 row=3, col=1)

fig.update_yaxes(title_text='Dur (h)', row=3, col=1)
fig.update_xaxes(title_text='Year', row=3, col=1)

# Show grid
fig.update_layout(
    xaxis_showgrid=True,
    yaxis_showgrid=True,
    showlegend=True,
    height=800,  # Adjust the height of the plot
    width=800,
    title_text="Storm Generation Matlab vs Python"
)

# Show plot
fig.show()

## Sea Level Rise

In [ ]:
import plotly.express as px

days = synthetic_storm['start'].apply(helper.calculate_days_since_2018)
days_m = synthetic_storm_m['date_start'].apply(helper.calculate_days_since_2018)
scenario = '0'

slr_values = slr.calculate_slr(days, scenario)
slr_values_m = slr.calculate_slr(days_m, scenario)
dates = [datetime(2018, 1, 1) + timedelta(days=int(day)) for day in days]

wl0 = 0 
synthetic_storm['slr'] = wl0 - slr_values[0] + slr_values
synthetic_storm_m['slr'] = wl0 - slr_values_m[0] + slr_values_m

fig = px.line(
    synthetic_storm,
    x='start', 
    y='slr', 
    labels={'start': 'Years', 'slr': 'Sea Level Rise (m)'},
    title=f'Sea Level Rise under {scenario} Scenario'
)

fig.show()

## Erosion

In [ ]:
import pcr.erosion as erosion
import plotly.express as px

delV, synthetic_storm['erosion_storm'] = erosion.mendoza(synthetic_storm)
_, synthetic_storm_m['erosion_storm'] = erosion.mendoza(synthetic_storm_m)


fig = px.scatter(
    synthetic_storm_m, 
    x='erosion_storm', # this one is from python
    y='del_x_mat', 
    labels={
        'erosion_storm': "Python", 
        'del_x_mat': 'Matlab'
    }
)

fig.update_layout(
    xaxis_showgrid=True,
    yaxis_showgrid=True,
    showlegend=True,
    height=600,  # Adjust the height of the plot
    width=600,
    title_text="Shroreline Retreat Matlab vs Python"
)

fig.show()

## Shoreline Evolution


In [ ]:
# calculate recovery
rec_rate = 7/365
synthetic_storm['recovery'] = shoreline.calculate_recovery(synthetic_storm, rec_rate)
synthetic_storm_m['recovery'] = shoreline.calculate_recovery(synthetic_storm_m, rec_rate)

# calculate retreat due to slr 
m = 0.024
synthetic_storm['slr_retreat'] = shoreline.calculate_slr_retreat(synthetic_storm, m)
synthetic_storm_m['slr_retreat'] = shoreline.calculate_slr_retreat(synthetic_storm_m, m)
# synthetic_storm_m['slr_retreat'] = 0.0

# track shoreline evolution 
shoreline_track = shoreline.track_shoreline(synthetic_storm)
shoreline_track_m = shoreline.track_shoreline(synthetic_storm_m)

In [ ]:
shoreline_track['time'] = helper.date_add_days(date_start, shoreline_track['day'])
shoreline_track_m['time'] = helper.date_add_days(date_start, shoreline_track_m['day'])

In [ ]:
import scipy.io 

mat_file = scipy.io.loadmat('../data/simulation_storm_m.mat')

matlab_shoreline_track = pd.DataFrame({
    'day': mat_file['day_plot'].flatten(),
    'x': mat_file['x_plot'].flatten()
})

matlab_shoreline_track = matlab_shoreline_track.dropna()

matlab_shoreline_track['time'] = helper.date_add_days(date_start, matlab_shoreline_track['day'])

In [ ]:
fig = plt.figure(figsize=(9,6))

plt.plot(shoreline_track_m['time'], shoreline_track_m['shoreline_position'])
plt.plot(matlab_shoreline_track['time'], matlab_shoreline_track['x'], '--')

plt.legend(['Python', 'Matlab'])

fig 

In [ ]:
matlab_shoreline_track = matlab_shoreline_track.dropna()

# Point of Interest 


In [ ]:
import cartopy.crs as ccrs
import geopandas as gpd 
import xarray as xr

In [ ]:
# read the feature collection 
poi_gdf = gpd.read_file('../data/point_of_interest.json')

# # rounding the coordinates
# def round_coord(coord): 
#     return round(coord*2) / 2

# gdf['geometry'] = gdf['geometry'].apply(lambda x: x.__class__(round_coord(x.x), round_coord(x.y)))

# plot the points 
# plt.figure(figsize=(8,6))

figpoi, ax = plt.subplots(1, 1, subplot_kw=dict(projection=ccrs.PlateCarree()))
# ax = plt.axes(projection=ccrs.PlateCarree())
ax.stock_img()
# ax.coastlines()
ax.set_title("area of interest")

poi_gdf.plot(ax=ax, marker='o', alpha=1, color='red')

for x, y, label in zip(poi_gdf.geometry.x, poi_gdf.geometry.y, poi_gdf.station):
    ax.annotate(label, xy=(x, y), xytext=(3, 3), textcoords='offset points')


In [ ]:
import xarray as xr

ds = xr.load_dataset('../data/test/reanalysis-era5-single-levels-timeseries-wav9u_yv630.nc')

In [ ]:
ds['mwp'].plot()

# Detect Storms ERA5 points 

In [ ]:
import xarray as xr
import scipy.io 

In [ ]:
nc_ds = xr.load_dataset('../data/ERA5/p7_1979.nc')
nc_ds.swh.plot()

In [ ]:
# we need hs, dir, tp, and time for detect
# however, we can just populate dir and tp with zeros for the meantime 

data_start = nc_ds.time[0].values
hs = nc_ds.squeeze().swh.values
time = (nc_ds.time.values - data_start).astype('timedelta64[h]').astype(float) / 24
dir = np.zeros(shape=hs.shape)
tp = np.zeros(shape=hs.shape)

ts_hs = 95
ts_dur = 12

storm_df, storm_ts = storm.detect(hs, dir, tp, time, ts_hs, ts_dur)

# create a histogram of percent frequency of occurance per month 
# get the month: day + day_start
hours = storm_df['start'].values * 24

storm_df['months'] = [(pd.to_datetime(data_start) + timedelta(hours=hour)).month for hour in hours]

monthly_count = storm_df[['hs_max', 'months']].groupby('months').count().rename(columns={
    'hs_max': 'p3'
})

monthly_perc = pd.DataFrame()
monthly_perc['p3'] = monthly_count['p3'] / monthly_count['p3'].sum() * 100

plt.bar(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], monthly_perc['p3'])
# plt.bar(monthly_perc.index, monthly_perc['p3'])

plt.ylim([0, 70])
plt.ylabel('Percent Frequency of Occurance')

In [ ]:
points = ['p3', 'p7', 'p11', 'p18', 'p20', 'p23']

ts_hs = 95
ts_dur = 12

monthly_perc = pd.DataFrame(
    index=np.arange(1, 13, 1)
)

monthly_count = pd.DataFrame(
    index=np.arange(1, 13, 1)
)

for point in points: 
    nc_df = xr.load_dataset(f'../data/ERA5/{point}_1979.nc')
    data_start = nc_df.time[0].values

    hs = nc_df.squeeze().swh.values
    time = (nc_df.time.values - data_start).astype('timedelta64[h]').astype(float) / 24
    dir = np.zeros(shape=hs.shape)
    tp = np.zeros(shape=hs.shape)

    storm_df, storm_ts = storm.detect(hs, dir, tp, time, ts_hs, ts_dur)

    hours = storm_df['start'].values * 24

    storm_df['months'] = [(pd.to_datetime(data_start) + timedelta(hours=hour)).month for hour in hours]

    monthly_count[point] = storm_df[['hs_max', 'months']].groupby('months').count().rename(columns={
        'hs_max': point
    })

    monthly_perc[point] = monthly_count[point] / monthly_count[point].sum() * 100

monthly_perc.replace(np.nan, 0, inplace=True)


In [ ]:
# import data from hector

data_fig3 = scipy.io.loadmat('../data/output/data_Figure3.mat')

nsws = data_fig3['Nsws']
nws = data_fig3['Nws']


In [ ]:
# create plots 

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
fig, axs = plt.subplots(2, 3, figsize=(18,8))
width = 0.4
x = np.arange(12)

axs[0,0].grid(linestyle='--', alpha=0.3, zorder=0)
axs[0,0].bar(x-width/2, monthly_perc['p3'], width, label= "PCR", zorder=3)
axs[0,0].bar(x+width/2, nws[:, 2], width, label="Lobeto et al., 2024", zorder=3)
axs[0,0].set_title('P3')
axs[0,0].set_ylim([0, 70])
# axs[0,0].legend(loc='upper right')
axs[0,0].set_xticks(x, months)
axs[0,0].set_ylabel('Percentage Frequency of Occurence')

axs[0,1].grid(linestyle='--', alpha=0.3, zorder=0)
axs[0,1].bar(x-width/2, monthly_perc['p7'], width, label= "PCR")
axs[0,1].bar(x+width/2, nws[:, 6], width, label="Lobeto et al., 2024")
axs[0,1].set_title('P7')
axs[0,1].set_ylim([0, 70])
# axs[0,1].legend(loc='upper right')
axs[0,1].set_xticks(x, months)
axs[0,1].set_ylabel('Percentage Frequency of Occurence')

axs[0,2].grid(linestyle='--', alpha=0.3, zorder=0)
axs[0,2].bar(x-width/2, monthly_perc['p11'], width, label= "PCR")
axs[0,2].bar(x+width/2, nws[:, 10], width, label="Lobeto et al., 2024")
axs[0,2].set_title('P11')
axs[0,2].set_ylim([0, 70])
# axs[0,2].legend(loc='upper right')
axs[0,2].set_xticks(x, months)
axs[0,2].set_ylabel('Percentage Frequency of Occurence')

axs[1,0].grid(linestyle='--', alpha=0.3, zorder=0)
axs[1,0].bar(x-width/2, monthly_perc['p18'], width, label= "PCR")
axs[1,0].bar(x+width/2, nws[:, 17], width, label="Lobeto et al., 2024")
axs[1,0].set_title('P18')
axs[1,0].set_ylim([0, 70])
# axs[1,0].legend(loc='upper right')
axs[1,0].set_xticks(x, months)
axs[1,0].set_ylabel('Percentage Frequency of Occurence')

axs[1,1].grid(linestyle='--', alpha=0.3, zorder=0)
axs[1,1].bar(x-width/2, monthly_perc['p20'], width, label='PCR')
axs[1,1].bar(x+width/2, nws[:, 19], width, label="Lobeto et al., 2024")
axs[1,1].set_title('P20')
axs[1,1].set_ylim([0, 70])
# axs[1,1].legend(loc='upper right')
axs[1,1].set_xticks(x, months)
axs[1,1].set_ylabel('Percentage Frequency of Occurence')

axs[1,2].grid(linestyle='--', alpha=0.3, zorder=0)
axs[1,2].bar(x-width/2, monthly_perc['p23'], width, label='PCR')
axs[1,2].bar(x+width/2, nws[:, 22], width, label="Lobeto et al., 2024")
axs[1,2].set_title('P23')
axs[1,2].set_ylim([0, 70])
# axs[1,2].legend(loc='upper right')
axs[1,2].set_xticks(x, months)
axs[1,2].set_ylabel('Percentage Frequency of Occurence')

fig.legend(['PCR', 'Lobeto et al., 2024'], loc='lower center', ncols=2)
fig.suptitle('Monthly percentage frequency of occurrence of wave storm events at key locations')